# Intrusion Detection with Logistic Regression
In this laboratory, we will use a Logistic Regression to classy the network traffic flows as benign or malicious. The Logistic regression model returns a value between 0 and 1, which is the probability of the input flow of being malicious. We use a threshold set to 0.5 to decide whether the network flow is malicious or not.
We will train a logistic regression model on a dataset of benign traffic and DDoS attack traffic.

We will use a dataset of benign and various DDoS attacks from the CIC-DDoS2019 dataset (https://www.unb.ca/cic/datasets/ddos-2019.html).
The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header features computed from max 10 packets:

| Features           | Logistic Regression model           |
|---------------------|--------------------|
| timestamp (mean IAT)  <br> packet_length (mean) <br> IP_ttl (mean) <br> IP_flags_df (sum) <br> IP_flags_mf (sum) <br> IP_flags_rb (sum) <br> IP_frag_off (sum) <br> TCP_length (mean) <br> TCP_flags_ack (sum) <br> TCP_flags_cwr (sum) <br> TCP_flags_ece (sum) <br> TCP_flags_fin (sum) <br> TCP_flags_push (sum) <br> TCP_flags_reset (sum) <br> TCP_flags_syn (sum) <br> TCP_flags_urg (sum) <br> TCP_window_size (mean) <br> UDP_length (mean) <br> ICMP_type (mean) <br> ICMP_code (mean) <br> Packets (counter) <br>| <img src="./logistic_regression_CIC2019.jpg" width="100%">  |

In [1]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Import necessary libraries

import numpy as np
import glob
import h5py
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import SGD,Adam
from util_functions import *
DATASET_FOLDER = "./DOS2019"

In [2]:
# Load training, validation and test sets
X_train, y_train = load_dataset(DATASET_FOLDER + "/*" + '-train.hdf5')
y_train = np.array([1 if y != 0 else 0 for y in y_train]) # from integer to binary

X_val, y_val = load_dataset(DATASET_FOLDER + "/*" + '-val.hdf5')
y_val = np.array([1 if y != 0 else 0 for y in y_val]) # from integer to binary

X_test, y_test = load_dataset(DATASET_FOLDER + "/*" + '-test.hdf5')
y_test = np.array([1 if y != 0 else 0 for y in y_test]) # from integer to binary

In [3]:
# Logistic Regression model
def LogRegression(model_name, input_shape):
    activation_function = 'sigmoid'
    model = Sequential(name=model_name)
    model.add(Dense(1, input_shape=input_shape,activation=activation_function, name='fc1'))

    print(model.summary())
    return model

# Optimisation algorithm
Here we can compare the performance of different optimizers 

In [4]:
def compileModel(model,lr):
    #optimizer = SGD(learning_rate=lr, momentum=0.0) # the optimisation algorithm
    optimizer = Adam(learning_rate=lr)
    model.compile(loss='binary_crossentropy', optimizer=optimizer,metrics=['accuracy'])  # here we specify the loss function

# Model training

In [5]:
model = LogRegression('log_reg', X_train.shape[1:4])
compileModel(model,0.001)

# Train the model
model.fit(X_train, y_train, epochs=200, batch_size=10, validation_data=(X_val, y_val))

/Users/roberto/miniconda3/envs/python313/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "log_reg"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc1 (Dense)                     │ (None, 1)              │            22 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22 (88.00 B)

 Trainable params: 22 (88.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7866 - loss: 0.8429 - val_accuracy: 0.7041 - val_loss: 0.8189
Epoch 2/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 443us/step - accuracy: 0.7866 - loss: 0.8190 - val_accuracy: 0.7041 - val_loss: 0.8008
Epoch 3/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 417us/step - accuracy: 0.7866 - loss: 0.7937 - val_accuracy: 0.7041 - val_loss: 0.7824
Epoch 4/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 424us/step - accuracy: 0.7866 - loss: 0.7683 - val_accuracy: 0.7041 - val_loss: 0.7641
Epoch 5/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 422us/step - accuracy: 0.7866 - loss: 0.7445 - val_accuracy: 0.7041 - val_loss: 0.7472
Epoch 6/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step - accuracy: 0.7866 - loss: 0.7218 - val_accuracy: 0.7041 - val_loss: 0.7281
Epoch 7/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 445us/step - accuracy: 0.7866 - loss: 0.7000 - val_accuracy: 0.7041 - val_loss: 0.7127
Epoch 8/200
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 432us/step - accuracy: 0.7866 - loss: 0.6800 - val_a

# Prediction on unseen data

In [6]:
y_pred = np.squeeze(model.predict(X_test, batch_size=32) > 0.5)

print("F1 Score: ", f1_score(y_test,y_pred))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
F1 Score:  0.9740932642487047
